<a href="https://colab.research.google.com/github/yeonsub/CRO/blob/main/ExaoneAgent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import requests
import json
import sys
from typing import Any, List, Optional, Dict
from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.messages import BaseMessage, AIMessage, HumanMessage, SystemMessage
from langchain_core.outputs import ChatResult, ChatGeneration
from langchain_core.tools import tool
from pydantic import Field
from google.colab import userdata

# 1. EXAONE Chat Model 정의
class ChatExaone(BaseChatModel):
    api_key: str = Field(default_factory=lambda: userdata.get('EXAONE_API_KEY'))
    model_name: str = "k-exaone_v2.5"
    api_url: str = "https://api.exaone.ai/v1/chat/completions"

    @property
    def _llm_type(self) -> str:
        return "exaone-chat"

    def _convert_messages_to_exaone_format(self, messages: List[BaseMessage]) -> List[Dict[str, str]]:
        exaone_messages = []
        for msg in messages:
            if isinstance(msg, SystemMessage):
                exaone_messages.append({"role": "system", "content": msg.content})
            elif isinstance(msg, HumanMessage):
                exaone_messages.append({"role": "user", "content": msg.content})
            elif isinstance(msg, AIMessage):
                exaone_messages.append({"role": "assistant", "content": msg.content})
            else:
                exaone_messages.append({"role": "user", "content": msg.content})
        return exaone_messages

    def _generate(
        self,
        messages: List[BaseMessage],
        stop: Optional[List[str]] = None,
        run_manager: Optional[Any] = None,
        **kwargs: Any,
    ) -> ChatResult:
        headers = {
            "Authorization": f"Bearer {self.api_key}",
            "Content-Type": "application/json"
        }
        payload = {
            "model": self.model_name,
            "messages": self._convert_messages_to_exaone_format(messages),
            "stream": False
        }
        response = requests.post(self.api_url, headers=headers, json=payload)
        response.raise_for_status()
        res_json = response.json()
        content = res_json['choices'][0]['message']['content']
        ai_message = AIMessage(content=content)
        return ChatResult(generations=[ChatGeneration(message=ai_message)])

# 2. 모델 인스턴스 생성
chat_model = ChatExaone()

# 3. 날씨 검색을 위한 도구(Tool) 정의
@tool
def get_current_weather_tool(location: str) -> str:
    """실시간 지정된 지역의 날씨 상태 및 예보 정보를 조회합니다."""
    if "대전" in location:
        return "대전 날씨: 18°C, 대체로 흐림. 오후 한때 비 소식 있음."
    else:
        return f"{location} 날씨: 20°C, 맑음."

print("EXAONE 연결 모델(chat_model)과 날씨 도구(get_current_weather_tool)가 준비되었습니다.")

EXAONE 연결 모델(chat_model)과 날씨 도구(get_current_weather_tool)가 준비되었습니다.


In [3]:
!pip install -q langgraph ipywidgets
import sys
from typing import Annotated, TypedDict, Literal
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_core.messages import BaseMessage, AIMessage, HumanMessage, SystemMessage
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# 1. LangGraph 상태(State) 설계
class AgentState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

# 2. 어시스턴트 판단 노드 (LLM 호출)
def assistant_node(state: AgentState):
    messages = state["messages"]
    system_prompt = SystemMessage(content="""당신은 다양하고 광범위한 주제에 대해 답변할 수 있는 유능한 범용 AI 어시스턴트입니다.
사용자의 질문에 맞춰 필요한 경우 제공된 도구를 사용해 정보를 찾고, 그렇지 않은 일반적인 질문이나 지식 기반 질문에는 직접 판단하여 즉시 최종 답변을 제공하세요.

날씨 정보 조회가 필요할 때는 답변에 반드시 '[날씨 조회 필요: 지역이름]' 이라는 특별한 문구를 포함하여 응답해 주세요.
그렇지 않은 일반적인 질문이나 대화는 도구 사용을 위한 언급 없이 친절하게 완성형 답변을 출력하세요.""")

    response = chat_model.invoke([system_prompt] + messages)
    return {"messages": [response]}

# 3. 날씨 정보 조회 도구 노드
def weather_tool_node(state: AgentState):
    messages = state["messages"]
    last_message = messages[-1]
    content = last_message.content

    location = "서울"
    for loc in ["대전", "부산", "제주", "인천", "서울"]:
        if loc in content:
            location = loc
            break

    tool_result = get_current_weather_tool.invoke({"location": location})
    return {"messages": [HumanMessage(content=f"실시간 도구 정보 확인 완료: {tool_result}. 이제 이 정보를 반영하여 사용자에게 친절하고 완벽한 최종 답변을 생성해줘.")]}

# 4. 조건부 라우터 에지 정의
def router_edge(state: AgentState) -> Literal["call_tool", "end"]:
    messages = state["messages"]
    last_message = messages[-1]

    if "[날씨 조회 필요" in last_message.content and not any("실시간 도구 정보 확인 완료" in m.content for m in reversed(messages)):
        return "call_tool"

    return "end"

# 5. 그래프 빌드 및 컴파일
workflow = StateGraph(AgentState)
workflow.add_node("assistant", assistant_node)
workflow.add_node("weather_tool", weather_tool_node)
workflow.add_edge(START, "assistant")
workflow.add_conditional_edges(
    "assistant",
    router_edge,
    {
        "call_tool": "weather_tool",
        "end": END
    }
)
workflow.add_edge("weather_tool", "assistant")
app = workflow.compile()

# 6. 대화형 Chat UI 구축 및 실행
chat_history_html = []
current_messages = []

output_box = widgets.Output()
text_input = widgets.Text(
    placeholder='여기에 질문을 입력하세요 (예: 서울 날씨 알려줘, 지구의 반지름은?)',
    layout=widgets.Layout(width='80%')
)
send_button = widgets.Button(
    description='전송',
    button_style='success',
    layout=widgets.Layout(width='18%', margin='0px 0px 0px 10px')
)

input_container = widgets.HBox([text_input, send_button])

def render_chat():
    with output_box:
        clear_output(wait=True)
        css_style = """
        <style>
            .chat-container { display: flex; flex-direction: column; max-height: 400px; overflow-y: auto; padding: 10px; background-color: #f7f9fa; border-radius: 8px; border: 1px solid #e1e4e8; }
            .bubble { padding: 10px 14px; margin: 5px 0; border-radius: 15px; max-width: 70%; line-height: 1.5; font-family: sans-serif; font-size: 14px; word-wrap: break-word; }
            .user-bubble { background-color: #007bff; color: white; align-self: flex-end; border-bottom-right-radius: 2px; }
            .ai-bubble { background-color: #e2e3e5; color: #333; align-self: flex-start; border-bottom-left-radius: 2px; }
            .system-bubble { background-color: #fff3cd; color: #856404; align-self: center; font-size: 12px; border-radius: 5px; text-align: center; }
        </style>
        """
        chat_content = "".join(chat_history_html)
        display(HTML(f"{css_style}<div class='chat-container'>{chat_content}</div>"))

def on_send_click(b):
    global current_messages
    query = text_input.value.strip()
    if not query:
        return

    text_input.value = ''
    text_input.disabled = True
    send_button.disabled = True

    chat_history_html.append(f"<div class='bubble user-bubble'>{query}</div>")
    render_chat()

    current_messages.append(HumanMessage(content=query))

    try:
        inputs = {"messages": current_messages}
        last_ai_content = ""

        for event in app.stream(inputs):
            for node_name, state_update in event.items():
                if "messages" in state_update:
                    last_msg = state_update["messages"][-1]
                    if node_name == "weather_tool":
                        chat_history_html.append(f"<div class='bubble system-bubble'>🔍 [시스템]: {last_msg.content.split('.')[0]}</div>")
                        render_chat()
                    else:
                        last_ai_content = last_msg.content

        if last_ai_content:
            formatted_ai = last_ai_content.replace('\n', '<br>')
            chat_history_html.append(f"<div class='bubble ai-bubble'>{formatted_ai}</div>")
            current_messages.append(AIMessage(content=last_ai_content))

    except Exception as e:
        chat_history_html.append(f"<div class='bubble system-bubble' style='background-color:#f8d7da; color:#721c24;'>⚠️ 에러 발생: {str(e)}</div>")

    render_chat()
    text_input.disabled = False
    send_button.disabled = False

send_button.on_click(on_send_click)
text_input.on_submit(on_send_click)

display(output_box)
display(input_container)

chat_history_html.append("<div class='bubble system-bubble'>🤖 EXAONE 에이전트가 활성화되었습니다. 대화를 시작해 보세요!</div>")
render_chat()

Output()